In [ ]:
import fastf1
from fastf1 import get_event_schedule, get_session
import pandas as pd
import os
import logging
import re

In [ ]:
# ========== CONFIG ==========
SEASON = 2026
WET_LAP_THRESHOLD = 0.1
cache_dir = '../.fastf1_cache'
output_dir = '../data/processed'
os.makedirs(cache_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

In [ ]:
fastf1.Cache.enable_cache(cache_dir)

compound_map = {
    'SOFT': 'Soft', 'MEDIUM': 'Medium', 'HARD': 'Hard',
    'INTERMEDIATE': 'Intermediate', 'WET': 'Wet'
}

def slugify(text):
    """Basic slugify for safe filenames."""
    return re.sub(r'[^\w]+', '_', text.lower()).strip('_')

In [ ]:
# ========== LOGGING ==========
logging.basicConfig(filename=f'fastf1_data_export_{SEASON}.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [ ]:
# ========== DATA START ==========
# NOTE: Only races that have already occurred will load successfully.
# Run this notebook after each race weekend to pick up new data.
schedule = get_event_schedule(SEASON, include_testing=False)
race_events = schedule.copy()
combined_export_path = f"{output_dir}/all_races_combined_{SEASON}.csv"

# Load existing data if the combined CSV already exists (incremental update)
if os.path.exists(combined_export_path):
    existing_df = pd.read_csv(combined_export_path, low_memory=False)
    already_imported = set(existing_df['GP_Slug'].unique())
    print(f"📂 Found existing file with {len(existing_df)} rows across {len(already_imported)} races.")
    print(f"   Already imported: {sorted(already_imported)}")
    all_races = [existing_df]
else:
    already_imported = set()
    all_races = []
    print("📂 No existing 2026 file — starting fresh.")

In [ ]:
for _, row in race_events.iterrows():
    round_num = row['RoundNumber']
    event_name = row['EventName']
    gp_name = slugify(event_name)
    event_date = row['EventDate']

    # Skip races already in the combined CSV
    if gp_name in already_imported:
        print(f"⏭️  Skipping (already imported): {event_name}")
        continue

    try:
        session = get_session(SEASON, round_num, 'R')
        session.load()
    except Exception as e:
        logging.error(f"[LOAD FAIL] {gp_name}: {e}")
        print(f"⏳ Not yet available or failed to load: {event_name} — {e}")
        continue

    try:
        laps = session.laps.reset_index(drop=True)
        selected_cols = [
            'Driver', 'Team', 'LapNumber', 'LapTime',
            'Sector1Time', 'Sector2Time', 'Sector3Time',
            'Compound', 'TyreLife', 'Stint',
            'PitInTime', 'PitOutTime', 'TrackStatus',
            'IsAccurate', 'Time'
        ]
        lap_data = laps[selected_cols].copy()

        for col in ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']:
            lap_data[f'{col}Seconds'] = lap_data[col].dt.total_seconds()
        lap_data['LapStartTime'] = lap_data['Time'].dt.total_seconds()

        # Weather Merge (Safe)
        try:
            weather = session.weather_data.rename(columns={'Time': 'WeatherTime'})
            weather['WeatherTime'] = weather['WeatherTime'].dt.total_seconds()
            lap_data = pd.merge_asof(
                lap_data.sort_values('LapStartTime'),
                weather.sort_values('WeatherTime'),
                left_on='LapStartTime', right_on='WeatherTime',
                direction='nearest'
            )
        except Exception:
            logging.warning(f"No weather data for {gp_name}")
            lap_data['Rainfall'] = 0
            lap_data['AirTemp'] = None
            lap_data['Humidity'] = None
            lap_data['WindSpeed'] = None

        # Rain flags
        lap_data['IsWetLap'] = lap_data['Rainfall'] > WET_LAP_THRESHOLD
        lap_data['IsDryLap'] = lap_data['Rainfall'] <= WET_LAP_THRESHOLD
        lap_data['IsWetRace'] = lap_data['IsWetLap'].any()

        # Circuit Metadata
        try:
            circuit_info = session.get_circuit_info()
            lap_data['CircuitName'] = circuit_info.name
            lap_data['CircuitShort'] = circuit_info.location
            lap_data['CircuitCountry'] = circuit_info.country
            lap_data['TrackLengthKM'] = circuit_info.length / 1000 if circuit_info.length else None
            lap_data['AltitudeM'] = circuit_info.altitude
        except Exception:
            event = session.event
            lap_data['CircuitName'] = event.get('OfficialEventName', event_name)
            lap_data['CircuitShort'] = event.get('Location', 'Unknown')
            lap_data['CircuitCountry'] = event.get('Country', 'Unknown')
            lap_data['TrackLengthKM'] = None
            lap_data['AltitudeM'] = None

        lap_data['CircuitType'] = lap_data['CircuitShort'].apply(lambda name: (
            "Street" if isinstance(name, str) and any(x in name.lower() for x in ['monaco', 'baku', 'miami', 'jeddah'])
            else "Hybrid" if isinstance(name, str) and 'marina' in name.lower()
            else "Permanent"
        ))

        # Type & Cleanup
        lap_data['TrackStatus'] = lap_data['TrackStatus'].astype(str)
        lap_data['IsAccurate'] = lap_data['IsAccurate'].astype(bool)
        lap_data['LapNumber'] = lap_data['LapNumber'].fillna(0).astype(int)
        lap_data['TyreLife'] = lap_data['TyreLife'].fillna(0).astype(int)
        lap_data['Stint'] = lap_data['Stint'].fillna(0).astype(int)

        lap_data.dropna(subset=[
            'LapTimeSeconds', 'Sector1TimeSeconds', 'Sector2TimeSeconds',
            'Sector3TimeSeconds', 'Compound'
        ], inplace=True)

        lap_data['Compound'] = lap_data['Compound'].str.upper().map(compound_map).fillna(lap_data['Compound'])

        # Pit Info
        lap_data['PitLap'] = lap_data.apply(
            lambda row: row['LapNumber'] if pd.notna(row['PitInTime']) else None, axis=1
        )
        lap_data['PitDuration'] = (lap_data['PitOutTime'] - lap_data['PitInTime']).dt.total_seconds()

        # Sector Features
        lap_data['Sector1Pct'] = lap_data['Sector1TimeSeconds'] / lap_data['LapTimeSeconds']
        lap_data['Sector2Pct'] = lap_data['Sector2TimeSeconds'] / lap_data['LapTimeSeconds']
        lap_data['Sector3Pct'] = lap_data['Sector3TimeSeconds'] / lap_data['LapTimeSeconds']
        lap_data['BestSector'] = lap_data[['Sector1TimeSeconds', 'Sector2TimeSeconds', 'Sector3TimeSeconds']].idxmin(axis=1)
        lap_data['BestSector'] = lap_data['BestSector'].str.extract(r'(\d)').astype(int)
        lap_data['IsValidLap'] = (lap_data['TrackStatus'] == 'Green') & lap_data['IsAccurate']

        # Metadata
        lap_data['GrandPrix'] = event_name
        lap_data['GP_Slug'] = gp_name
        lap_data['SeasonYear'] = SEASON
        lap_data['EventDate'] = pd.to_datetime(event_date)

        if 'CarNumber' in laps.columns:
            lap_data['CarNumber'] = laps['CarNumber']

        # Stint Summary
        stint_summary = lap_data.groupby(['Driver', 'Stint']).agg(
            AvgLapTime=('LapTimeSeconds', 'mean'),
            StintLength=('LapNumber', 'count')
        ).reset_index()
        lap_data = lap_data.merge(stint_summary, on=['Driver', 'Stint'], how='left')

        stint_max_map = lap_data.groupby('Driver')['Stint'].max().to_dict()
        lap_data['StintType'] = lap_data.apply(
            lambda row: "Opening" if row['Stint'] == 1 else
                        "Closing" if row['Stint'] == stint_max_map.get(row['Driver'], 3) else "Mid", axis=1
        )

        # Delta to Driver's Fastest Lap
        fastest_per_driver = lap_data.groupby("Driver")["LapTimeSeconds"].min().to_dict()
        lap_data['DeltaToFastestLap'] = lap_data.apply(
            lambda row: row["LapTimeSeconds"] - fastest_per_driver.get(row["Driver"], row["LapTimeSeconds"]),
            axis=1
        )

        # Flags
        lap_data["IsSC"] = lap_data["TrackStatus"].str.contains("4").fillna(False)
        lap_data["IsVSC"] = lap_data["TrackStatus"].str.contains("8").fillna(False)
        lap_data["IsRedFlag"] = lap_data["TrackStatus"].str.contains("16").fillna(False)
        race_max_lap = lap_data["LapNumber"].max()
        lap_data["IsDNF"] = lap_data["LapNumber"] < (race_max_lap - 3)

        # Export individual race CSV
        race_csv = f"{output_dir}/race_summary_{SEASON}_{gp_name}.csv"
        lap_data.to_csv(race_csv, index=False)
        all_races.append(lap_data)
        already_imported.add(gp_name)
        logging.info(f"[EXPORT OK] {gp_name}")
        print(f"✅ Exported: {event_name} ({len(lap_data)} laps)")

    except Exception as e:
        logging.error(f"[PROCESS FAIL] {gp_name}: {e}")
        print(f"❌ Failed processing for {event_name} — {e}")

In [ ]:
# Combined Export — rewrites the file with all races (old + new)
if all_races:
    combined_df = pd.concat(all_races, ignore_index=True)
    combined_df.to_csv(combined_export_path, index=False)
    print(f"\n🎉 Combined season export → {combined_export_path}")
    print(f"   Total rows: {len(combined_df)} across {combined_df['GP_Slug'].nunique()} races")
else:
    print("⚠️ No races were processed.")

In [ ]:
# ========== IMPORT NEW RACES TO SUPABASE ==========
# Run this cell after the notebook exports new race data to push it to the DB.
import sys
sys.path.insert(0, '../backend')

from scripts.import_csv_to_db import import_season_data
from app.core.config import settings
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from app.models.database import Base

engine = create_engine(settings.DATABASE_URL)
Base.metadata.create_all(bind=engine)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
session = SessionLocal()

try:
    import_season_data(combined_export_path, session, SEASON)
    print("\n✅ Supabase DB updated with latest 2026 races!")
finally:
    session.close()